# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. My rule and its reason codes

I will prioritize content that has meaningful historical search visibility and shows an opportunity for review based on its historical CTR.

The baseline rule uses three pre-decision signals:

1. **Historical impressions (`imp_prev45`)** — higher historical search volume means a page has more observed visibility and therefore potentially more value to review.
2. **Historical CTR (`clk_prev45 / imp_prev45`)** — lower CTR indicates that the page is receiving impressions but generating relatively fewer clicks, which provides a reason to inspect the result.
3. **Visible query count (`visible_queries`)** — broader query coverage indicates that the content has a wider historical search footprint.

Before encoding the rule, I audit two signals using buckets:

- Historical impressions / volume, which is linked to the FlyRank quick-win volume logic.
- Historical CTR, which is linked to the FlyRank CTR-fix logic.

For each signal I report the number of observations (`n`) in each bucket and compare the observed decline rate across buckets. The verdict is one of:

- `CONFIRMED` — the observed relationship supports the signal direction.
- `OPPOSITE` — the observed relationship is consistently opposite to the expected direction.
- `MIXED` — the relationship is inconsistent or weak.
- `FALSE` — the signal does not provide useful evidence for the intended rule.

These signal checks are validation of the rule idea. The future-window decline label is not used as an input to the baseline score.

### Baseline rule

The baseline score combines:

- 50% historical visibility
- 30% query breadth
- 20% CTR opportunity

Higher historical impressions and broader query coverage increase the priority score. Lower historical CTR increases the priority score.

The rule produces one reason code and one action label:

**Reason code:** `VISIBILITY_REVIEW`

**Action:** `REVIEW_AND_REFRESH`

The baseline is intended as decision support rather than a prediction or causal model. It uses only information available before the decision point and does not use future-window measurements, the decline label, or product-generated flags as scoring inputs.

In [16]:
# ============================================================
# ML-07 — SECTION 1
# My Rule and Its Reason Codes
# ============================================================

import duckdb
import pandas as pd
import numpy as np
import os
import getpass


# ============================================================
# 1. DUCKDB + HUGGING FACE SETUP
# ============================================================

if "con" not in globals():
    con = duckdb.connect()

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF Token: ")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required.")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
    "fact_query_90d": (
        f"read_parquet("
        f"'{REL}/fact_content_query_90d.parquet'"
        f")"
    ),
}

print("DuckDB connected.")
print("FlyRank warehouse configured.")


# ============================================================
# 2. CHECK DAILY DATA RANGE
# ============================================================

date_range = con.execute(
    f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES["fact_daily"]}
    """
).df()

print("\nAvailable daily data range:")
display(date_range)

max_date = pd.Timestamp(
    date_range.loc[0, "max_date"]
)

if pd.isna(max_date):
    raise ValueError(
        "Could not determine the latest report_date."
    )


# ============================================================
# 3. DEFINE THE 45-DAY OBSERVATION WINDOW
# ============================================================

# We use the latest 90 available days:
#
# Previous 45 days = historical decision-time window
# Last 45 days    = future evaluation window
#
# The future window is used ONLY to create the validation label.
# It is NOT used by the baseline score.

previous_start = (
    max_date - pd.Timedelta(days=89)
)

previous_end = (
    max_date - pd.Timedelta(days=45)
)

future_start = (
    max_date - pd.Timedelta(days=44)
)

future_end = max_date

print("\nML-07 windows:")
print(
    f"Historical window: "
    f"{previous_start.date()} -> {previous_end.date()}"
)

print(
    f"Future window:     "
    f"{future_start.date()} -> {future_end.date()}"
)


# ============================================================
# 4. BUILD CONTENT-LEVEL DAILY FEATURES
# ============================================================

df = con.execute(
    f"""
    WITH content_daily AS (

        SELECT
            client_hash_id,
            content_hash_id,
            report_date,

            COALESCE(gsc_impressions, 0)
                AS gsc_impressions,

            COALESCE(gsc_clicks, 0)
                AS gsc_clicks,

            gsc_avg_position

        FROM {TABLES["fact_daily"]}

        WHERE report_date
              BETWEEN DATE '{previous_start.date()}'
              AND DATE '{future_end.date()}'
    ),

    aggregated AS (

        SELECT

            client_hash_id,
            content_hash_id,

            -- Historical 45-day impressions
            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{previous_start.date()}'
                         AND DATE '{previous_end.date()}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev45,

            -- Historical 45-day clicks
            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{previous_start.date()}'
                         AND DATE '{previous_end.date()}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_prev45,

            -- Historical average position
            AVG(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{previous_start.date()}'
                         AND DATE '{previous_end.date()}'
                    THEN gsc_avg_position
                    ELSE NULL
                END
            ) AS pos_prev45,

            -- Future 45-day impressions
            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{future_start.date()}'
                         AND DATE '{future_end.date()}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_last45,

            -- Future 45-day clicks
            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{future_start.date()}'
                         AND DATE '{future_end.date()}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_last45

        FROM content_daily

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT *
    FROM aggregated
    """
).df()

print(
    f"\nContent-level historical dataset created: "
    f"{len(df):,} rows"
)


# ============================================================
# 5. ADD VISIBLE QUERY COUNT
# ============================================================

query_features = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(
            content_visible_query_count
        ) AS visible_queries

    FROM {TABLES["fact_query_90d"]}

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()


df = df.merge(
    query_features,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="left"
)

df["visible_queries"] = (
    pd.to_numeric(
        df["visible_queries"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# 6. CREATE FUTURE DECLINE LABEL
# ============================================================

# IMPORTANT:
# This label is ONLY for signal validation.
# It will NOT be used in the baseline score.

df["is_declining"] = (
    df["imp_last45"]
    < 0.80 * df["imp_prev45"]
).astype(int)


# ============================================================
# 7. HISTORICAL CTR
# ============================================================

df["historical_ctr"] = np.where(
    df["imp_prev45"] > 0,
    df["clk_prev45"] / df["imp_prev45"],
    np.nan
)


# ============================================================
# 8. KEEP ELIGIBLE CONTENT
# ============================================================

audit_df = df[
    df["imp_prev45"] > 0
].copy()

print(
    f"Eligible content with historical impressions: "
    f"{len(audit_df):,}"
)


# ============================================================
# 9. HISTORICAL IMPRESSIONS / VOLUME AUDIT
# ============================================================

audit_df["volume_bucket"] = pd.qcut(
    audit_df["imp_prev45"],
    q=4,
    duplicates="drop"
)

volume_audit = (
    audit_df
    .groupby(
        "volume_bucket",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("is_declining", "mean")
    )
    .reset_index()
)

volume_audit["decline_rate"] *= 100

print("\nHistorical impressions audit:")
display(volume_audit)


# ============================================================
# 10. HISTORICAL CTR AUDIT
# ============================================================

audit_df["ctr_bucket"] = pd.qcut(
    audit_df["historical_ctr"],
    q=4,
    duplicates="drop"
)

ctr_audit = (
    audit_df
    .groupby(
        "ctr_bucket",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("is_declining", "mean")
    )
    .reset_index()
)

ctr_audit["decline_rate"] *= 100

print("\nHistorical CTR audit:")
display(ctr_audit)


# ============================================================
# 11. SIGNAL VERDICT FUNCTION
# ============================================================

def get_signal_verdict(
    rates,
    expected_direction
):

    rates = np.asarray(
        rates,
        dtype=float
    )

    rates = rates[
        ~np.isnan(rates)
    ]

    if len(rates) < 2:
        return "FALSE"

    differences = np.diff(rates)

    increasing = np.sum(
        differences > 0
    )

    decreasing = np.sum(
        differences < 0
    )

    total = len(differences)

    if total == 0:
        return "FALSE"

    increasing_share = (
        increasing / total
    )

    decreasing_share = (
        decreasing / total
    )

    if expected_direction == "increasing":

        if increasing_share >= 0.67:
            return "CONFIRMED"

        if decreasing_share >= 0.67:
            return "OPPOSITE"

        return "MIXED"

    if expected_direction == "decreasing":

        if decreasing_share >= 0.67:
            return "CONFIRMED"

        if increasing_share >= 0.67:
            return "OPPOSITE"

        return "MIXED"

    return "FALSE"


# ============================================================
# 12. SIGNAL VERDICTS
# ============================================================

volume_verdict = get_signal_verdict(
    volume_audit["decline_rate"].values,
    expected_direction="increasing"
)

ctr_verdict = get_signal_verdict(
    ctr_audit["decline_rate"].values,
    expected_direction="decreasing"
)


# ============================================================
# 13. SIGNAL SUMMARY
# ============================================================

signal_summary = pd.DataFrame({
    "signal": [
        "Historical impressions",
        "Historical CTR"
    ],

    "expected_direction": [
        "Higher impressions -> higher decline rate",
        "Lower CTR -> higher decline rate"
    ],

    "verdict": [
        volume_verdict,
        ctr_verdict
    ]
})

print("\nSignal audit summary:")
display(signal_summary)


# ============================================================
# 14. BASELINE RULE CONFIGURATION
# ============================================================

BASELINE_WEIGHTS = {
    "visibility": 0.50,
    "query_breadth": 0.30,
    "ctr_opportunity": 0.20
}

REASON_CODE = "VISIBILITY_REVIEW"

ACTION = "REVIEW_AND_REFRESH"

print("\nBaseline rule configured:")
print(
    f"Historical visibility: "
    f"{BASELINE_WEIGHTS['visibility']:.0%}"
)

print(
    f"Query breadth: "
    f"{BASELINE_WEIGHTS['query_breadth']:.0%}"
)

print(
    f"CTR opportunity: "
    f"{BASELINE_WEIGHTS['ctr_opportunity']:.0%}"
)

print(
    f"Reason code: {REASON_CODE}"
)

print(
    f"Action: {ACTION}"
)


# ============================================================
# 15. FINAL SECTION 1 CHECK
# ============================================================

required_columns = [
    "content_hash_id",
    "imp_prev45",
    "clk_prev45",
    "visible_queries",
    "historical_ctr",
    "imp_last45",
    "clk_last45",
    "is_declining"
]

missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Section 1 is missing columns: {missing}"
    )

print("\n" + "=" * 60)
print("SECTION 1 COMPLETE")
print("=" * 60)
print(f"Total content items: {len(df):,}")
print(f"Eligible items:      {len(audit_df):,}")
print(f"Volume verdict:      {volume_verdict}")
print(f"CTR verdict:         {ctr_verdict}")
print("Baseline inputs:")
print("  - imp_prev45")
print("  - clk_prev45")
print("  - visible_queries")
print("Future label used only for validation: is_declining")
print("=" * 60)

HF Token: ··········
DuckDB connected.
FlyRank warehouse configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Available daily data range:


,min_date,max_date
0,2025-01-27,2026-06-30



ML-07 windows:
Historical window: 2026-04-02 -> 2026-05-16
Future window:     2026-05-17 -> 2026-06-30


...


Historical CTR audit:


,ctr_bucket,n,decline_rate
0,"(-0.001, 0.00204]",159735,67.643910
1,"(0.00204, 1.0]",53238,56.127202



Signal audit summary:


,signal,expected_direction,verdict
0,Historical impressions,Higher impressions -> higher decline rate,MIXED
1,Historical CTR,Lower CTR -> higher decline rate,CONFIRMED



Baseline rule configured:
Historical visibility: 50%
Query breadth: 30%
CTR opportunity: 20%
Reason code: VISIBILITY_REVIEW
Action: REVIEW_AND_REFRESH

SECTION 1 COMPLETE
Total content items: 409,326
Eligible items:      212,973
Volume verdict:      MIXED
CTR verdict:         CONFIRMED
Baseline inputs:
  - imp_prev45
  - clk_prev45
  - visible_queries
Future label used only for validation: is_declining


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
# ============================================================
# ML-07 — SECTION 2
# Build the ranked baseline action queue
# ============================================================

# Work on a copy so the original dataframe remains unchanged.
work = df.copy()


# ------------------------------------------------------------
# Required historical signals
# ------------------------------------------------------------

required = [
    "content_hash_id",
    "imp_prev45",
    "clk_prev45",
    "visible_queries"
]

missing = [
    col for col in required
    if col not in work.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )


# ------------------------------------------------------------
# Clean historical values
# ------------------------------------------------------------

work["imp_prev45"] = pd.to_numeric(
    work["imp_prev45"],
    errors="coerce"
)

work["clk_prev45"] = pd.to_numeric(
    work["clk_prev45"],
    errors="coerce"
)

work["visible_queries"] = pd.to_numeric(
    work["visible_queries"],
    errors="coerce"
)


# ------------------------------------------------------------
# Only content with historical impressions is eligible.
# ------------------------------------------------------------

work = work[
    work["imp_prev45"] > 0
].copy()


# ------------------------------------------------------------
# Historical CTR
# ------------------------------------------------------------

work["historical_ctr"] = (
    work["clk_prev45"].fillna(0)
    / work["imp_prev45"]
)


# ------------------------------------------------------------
# Convert signals to percentile scores
# ------------------------------------------------------------

# Higher historical impressions = higher priority.
work["visibility_score"] = (
    work["imp_prev45"].rank(pct=True)
)

# More visible queries = broader search footprint.
work["query_breadth_score"] = (
    work["visible_queries"].rank(pct=True)
)

# Lower historical CTR = greater review opportunity.
work["ctr_opportunity_score"] = (
    1 - work["historical_ctr"].rank(pct=True)
)


# ------------------------------------------------------------
# Final baseline score
# ------------------------------------------------------------

work["score"] = (
    0.50 * work["visibility_score"]
    + 0.30 * work["query_breadth_score"]
    + 0.20 * work["ctr_opportunity_score"]
)


# ------------------------------------------------------------
# Reason code and action
# ------------------------------------------------------------

work["reason_code"] = "VISIBILITY_REVIEW"

work["action"] = "REVIEW_AND_REFRESH"


# ------------------------------------------------------------
# Build ranked queue
# ------------------------------------------------------------

queue = (
    work[
        [
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "imp_prev45",
            "visible_queries",
            "historical_ctr"
        ]
    ]
    .sort_values(
        by=["score", "content_hash_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Add rank
# ------------------------------------------------------------

queue.insert(
    0,
    "rank",
    range(1, len(queue) + 1)
)


# ------------------------------------------------------------
# Write required CSV
# ------------------------------------------------------------

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(
    "work/outputs",
    exist_ok=True
)

queue.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# Section 2 output
# ------------------------------------------------------------

print("Baseline queue created successfully.")
print(f"Eligible content items: {len(queue):,}")
print(f"CSV written to: {output_path}")

print("\nTop 20 baseline picks:")
display(queue.head(20))

Baseline queue created successfully.
Eligible content items: 212,973
CSV written to: work/outputs/baseline_action_score.csv

Top 20 baseline picks:


,rank,content_hash_id,score,reason_code,action,imp_prev45,visible_queries,historical_ctr
0,1,content_3d611a5bb49884fa,0.935361,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,46466.0,301.0,0.0
1,2,content_066c2291283a5200,0.934120,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,35737.0,315.0,0.0
2,3,content_dda8be96d809ab55,0.931797,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,32204.0,164.0,0.0
3,4,content_1ff3c48911f11e70,0.930722,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,26330.0,187.0,0.0
4,5,content_17b9e821ceca190e,0.929692,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,23140.0,194.0,0.0
5,6,content_644828d3eca4ee75,0.925613,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,19216.0,191.0,0.0
6,7,content_78a3af7fee2ecbd7,0.925538,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,19590.0,174.0,0.0
7,8,content_1d7764b642f7bb9f,0.924918,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,17622.0,231.0,0.0
8,9,content_8d11fa8b52cf228f,0.922600,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,17733.0,129.0,0.0
9,10,content_8e1334d6356668e3,0.921333,VISIBILITY_REVIEW,REVIEW_AND_REFRESH,135914.0,47.0,0.0


## 3. Top-20 review

The top 20 items are all assigned the same baseline action, `REVIEW_AND_REFRESH`, with the reason code `VISIBILITY_REVIEW`. The ranking is based only on historical impressions, visible query breadth, and historical CTR.

All 20 top-ranked items have a historical CTR of `0.0`, so the CTR opportunity component contributes strongly to their high priority scores. Their historical impressions range from 13,265 to 135,914, while visible query counts range from 47 to 422.

| Rank | Historical impressions | Visible queries | Historical CTR | Action               | Reason code         | Confidence note                                                                                | What would make it wrong                                                                       |
| ---: | ---------------------: | --------------: | -------------: | -------------------- | ------------------- | ---------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------- |
|    1 |                 46,466 |             301 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | High baseline priority because visibility and query breadth are both substantial and CTR is 0. | The page may have valid search intent or tracking conditions that explain the zero clicks.     |
|    2 |                 35,737 |             315 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | High priority from strong visibility, broad query coverage, and zero historical CTR.           | Query intent or measurement issues could make the low CTR less actionable.                     |
|    3 |                 32,204 |             164 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Strong visibility and moderate query breadth support review.                                   | The impressions may come from queries where clicks are not expected.                           |
|    4 |                 26,330 |             187 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Good historical visibility with a clear CTR opportunity signal.                                | Zero clicks may reflect SERP context or mismatched query intent rather than weak content.      |
|    5 |                 23,140 |             194 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Meaningful visibility and query breadth make this a reasonable review candidate.               | The observed impressions may not represent commercially useful searches.                       |
|    6 |                 19,216 |             191 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Moderate visibility and broad query coverage support the action.                               | The page may already be appropriate for its search intent despite zero clicks.                 |
|    7 |                 19,590 |             174 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Similar to rank 6, with substantial historical visibility and zero CTR.                        | The baseline does not observe SERP features or query intent.                                   |
|    8 |                 17,622 |             231 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Broad query coverage strengthens the review priority.                                          | Many impressions may come from low-value or poorly matched queries.                            |
|    9 |                 17,733 |             129 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Meaningful visibility combined with zero CTR gives a clear review signal.                      | The zero CTR may be caused by the type of search result rather than the content.               |
|   10 |                135,914 |              47 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Very high visibility makes this a high-value review candidate despite narrower query breadth.  | High impressions may dominate the score even when the page has limited query coverage.         |
|   11 |                 18,360 |             101 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Moderate visibility and zero CTR support review.                                               | The page may have legitimate reasons for receiving impressions without clicks.                 |
|   12 |                 13,281 |             422 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Extremely broad query coverage and zero CTR make this worth inspection.                        | Broad query coverage does not guarantee that the queries are relevant to the page.             |
|   13 |                 16,030 |             129 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Meaningful visibility and zero CTR support prioritization.                                     | Search intent or SERP structure could explain the result.                                      |
|   14 |                 15,328 |             138 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Moderate visibility, query coverage, and zero CTR indicate a review opportunity.               | The observed impressions may not translate into actionable traffic.                            |
|   15 |                 20,006 |              82 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Higher visibility provides a reasonable basis for review.                                      | Lower query breadth means the baseline has less evidence about the breadth of the opportunity. |
|   16 |                 14,869 |             121 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Moderate visibility and zero CTR support review.                                               | The page could be correctly positioned for its queries despite low clicks.                     |
|   17 |                 14,596 |             121 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Similar historical footprint and zero CTR make review reasonable.                              | The baseline cannot determine whether refreshing the content will improve clicks.              |
|   18 |                 13,265 |             153 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Query breadth is relatively strong compared with its visibility, while CTR is zero.            | The query set may contain searches where clicks are not the expected action.                   |
|   19 |                 15,670 |             100 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Moderate visibility and zero CTR make this a reasonable review candidate.                      | The low CTR may reflect search-result behavior outside the content itself.                     |
|   20 |                 14,539 |             110 |         0.0000 | `REVIEW_AND_REFRESH` | `VISIBILITY_REVIEW` | Moderate historical visibility and query breadth support review.                               | The baseline does not account for intent, SERP features, or content quality.                   |

### Overall review

The top 20 are **priority candidates, not confirmed problems**. Their high scores are mainly driven by meaningful historical visibility, query coverage, and zero historical CTR.

The strongest potential weak pick is rank 10. It has the highest historical impressions in the top 20 but only 47 visible queries. This shows how the 50% visibility weight can push a highly visible page near the top even when its query breadth is relatively narrow.

Rank 12 is an opposite type of case: it has the highest visible query count in the top 20 but comparatively low historical impressions. Its high ranking shows the contribution of the 30% query-breadth component.

The baseline should therefore be treated as a **review queue**, not proof that every selected page needs a refresh.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

A few picks look weaker than others because the baseline is a simple rule and does not consider search intent, SERP features, or content quality. Rank 10 is the clearest example: it has very high historical impressions (135,914) but only 47 visible queries, so its high score is driven strongly by the 50% visibility component. Rank 12 is another edge case: it has the highest visible query count (422) but relatively low historical impressions (13,281). These are useful review candidates, but neither should automatically be treated as a confirmed refresh need.

The baseline does not use product-generated flags or future-window measurements as scoring inputs. The score uses only imp_prev45, visible_queries, and historical CTR calculated from clk_prev45 / imp_prev45. The future imp_last45 and clk_last45 are used only to create is_declining for signal validation in Section 1, not to calculate the baseline score. This follows the assignment's separation between pre-decision features and future evaluation data.

Leakage check: No product flags were included in the baseline score. No future-window measurements were included in the score. The is_declining label was created from the future window only for validation and was not used as an input to ranking.

Verdict: The baseline is a decision-support queue, not a prediction of which pages will decline. The main weakness is that the simple rule can over-prioritize pages with extreme visibility or query breadth, so the top picks should be manually reviewed before action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.